## A/B Testing

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# only predicted churn customers
ab_df = result_df[result_df["predicted_churn"] == 1].copy()

In [ ]:
import numpy as np
import pandas as pd

# only predicted churn customers
ab_df = result_df[result_df["predicted_churn"] == 1].copy()

# random A/B split
np.random.seed(42)
ab_df["group"] = np.random.choice(
    ["Control", "Treatment"],
    size=len(ab_df),
    p=[0.5, 0.5]
)

# assumptions
control_retention_rate = 0.20
treatment_retention_rate = 0.45
intervention_cost = 5

# simulate retained or not
ab_df["retained_after_campaign"] = ab_df["group"].apply(
    lambda x: np.random.binomial(
        1,
        treatment_retention_rate if x == "Treatment" else control_retention_rate
    )
)

# cost only applies to treatment group
ab_df["intervention_cost"] = np.where(
    ab_df["group"] == "Treatment",
    intervention_cost,
    0
)

# revenue saved only if retained
ab_df["revenue_saved"] = (
    ab_df["retained_after_campaign"] * ab_df["monthly_revenue"]
)

ab_df["net_gain"] = (
    ab_df["revenue_saved"] - ab_df["intervention_cost"]
)

ab_summary = ab_df.groupby("group").agg(
    customers=("customer_id", "count"),
    retained=("retained_after_campaign", "sum"),
    revenue_saved=("revenue_saved", "sum"),
    intervention_cost=("intervention_cost", "sum"),
    net_gain=("net_gain", "sum")
).reset_index()

ab_summary["retention_rate"] = (
    ab_summary["retained"] / ab_summary["customers"] * 100
)

ab_summary